## 1. Exercise: Study & Run the repository's multi-agent graph

Your first exercise will be to load, study and run the multi-agent graph provided in this repository.
The multi-agent graph is based on [pydantic-ai](https://pydantic.dev/pydantic-ai) (to set up the agents and tools) and
[pydantic-graph](https://pydantic.dev/docs/ai/graph/graph/) (to create the multi-agent architecture).

Let's load the environment variables. They are automatically set up in the `settings`-instance imported below.
In the environment variables, we define the OpenAI API and Tavily API keys as well as the Langfuse credentials.

In [ ]:
# ruff: noqa: E402
import sys
from pathlib import Path

# Construct src-path and append it to sys.path
src_path = Path.cwd().parent.parent
sys.path.append(str(src_path))

# NOTE: Initialize settings-object that loads the OpenAI API and Tavily API keys + langfuse credentials to the environment
from core.settings import settings  # noqa: F401

## 1.1 Load & Print the multi-agent graph

To begin with, let's analyze the multi-agent architecture by loading the `graph`-instance from this repository and print
a mermaid-diagram to analyze its structure.

&#10071; **Questions:**
- What nodes are contained?
- Which nodes are the entry points?
- Which nodes are connected?
- Which nodes are the end points?
- What are the roles of the PlannerNode and ExecutorNode?

<div class="alert alert-block alert-info">
<b>Tip:</b> To better understand the different roles of the nodes/agents in the graph, check out the implementations
of the different nodes in the folder `src/agents/&lt;name_of_agent&gt;/&lt;name_of_agent&gt;_node.py`.
</div>

In [ ]:
from IPython.display import Image

from agents import PlannerNode
from graph import graph

mermaid = Image(graph.mermaid_image(start_node=PlannerNode))
display(mermaid)

## 1.2 Run the multi-agent graph

Next, let us define the input arguments to run the multi-agent graph.

### Initiate the agent's initial state

First, let us specify the input parameters: the user query and the list of enabled agents.

In [ ]:
from models.agents import Agents

# Generally the user query should be solvable with the list of enabled agents below
user_query = "What is the current market capitalization of the top bank in Germany? State the day to which the final figure refers."

# List enabled agents that should be activated in the multi-agent graph
# Per default, the Planner & Executor are always activated
enabled_agents = [Agents.WEB_RESEARCHER, Agents.SYNTHESIZER]

Next, we will initiate the initial *state* of the agent.
The *state* provides the agent with a shared, evolving memory across nodes so that the agent has the context
and instructions needed to act coherently and achieve the goal.

<div class="alert alert-block alert-info">
<b>Tip:</b> Let's check out the `State`-object in `src/models/state.py` to learn more about it, i.e. see the attributes
defining each `State`-instance.
</div>

In [ ]:
from models.state import Message, State

state = State(
    messages=[Message(content=user_query)],
    user_query=user_query,
    enabled_agents=enabled_agents,
)

### Run the multi-agent graph

Execute the following cell to run the graph. This might take several seconds to minutes -- depending on the number of web searches executed by the multi-agent.
If interested, check out the logs printed out during execution.

In [ ]:
from agents.planner import PlannerNode
from graph import graph

# NOTE: When running the graph, we need to specify the starting point (here, PlannerNode) and the initial state
await graph.run(start_node=PlannerNode(), state=state)

### Review the agents' results

First, let's check out the multi-agent graph's final answer. Verify if it is correct.

In [ ]:
print(state.messages[-1].content)

Second, let's check out the plan created by the `PlannerNode` to answer the user's question. Is the plan reasonable, and does it align with the final answer?

In [ ]:
plan_pretty_out = [
    f"{_step.step_id} - ({_step.instruction.agent}): {_step.instruction.action}\n" for _step in state.plan.steps
]

print(*plan_pretty_out, sep="")